In [ ]:
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import spacy

from dap_job_quality.utils.keyword_search_patterns import keywords
from dap_job_quality.getters.ojo_getters import get_ojo_sample
from dap_job_quality.utils.spacy_keyword_search import get_matches, get_spans
from dap_job_quality.utils.text_cleaning import clean_text

from dap_job_quality.getters.data_getters import load_s3_jsonl
from dap_job_quality.getters.labelled_data import get_labelled_job_sentences
from dap_job_quality.utils import prodigy_data_utils as pdu

from dap_job_quality import BUCKET_NAME, PROJECT_DIR, config

model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
nlp = spacy.load("en_core_web_sm")

SEED = config["seed"]

def get_negative_example_sentences(df):
    # Find all unique texts
    unique_texts = df["text"].unique()

    # Split unique texts into sentences
    all_sentences_from_text = set()
    for text in unique_texts:
        doc = nlp(text)
        all_sentences_from_text.update([sent.text for sent in doc.sents])

    # Set of sentences already in the 'sentence' column
    existing_sentences = set(df["sentence"])

    # Find sentences that are not in the 'sentence' column
    new_sentences = all_sentences_from_text - existing_sentences

    return new_sentences


def filter_job_ads(labelled_df):
    job_ids = labelled_df["id"].unique()

    # skip the first 10 job ads - we didn't know what we were labelling at that point
    target_ids = job_ids[10:]

    labelled_df_clean = labelled_df[labelled_df["id"].isin(target_ids)]
    # get rid of empty spans
    labelled_df_clean = labelled_df_clean[labelled_df_clean["span"] != ""]
    return labelled_df_clean

In [ ]:
labelled_sents = get_labelled_job_sentences()[0]

labelled_data = pdu.get_spans_and_sentences(labelled_sents)

labelled_df = pd.DataFrame(columns=["span", "sent", "text", "job_id"])

for key in labelled_data.keys():
    temp_df = pd.DataFrame(labelled_data[key])
    temp_df["id"] = int(key)
    labelled_df = pd.concat([labelled_df, temp_df])

labelled_df = labelled_df.drop(["job_id"], axis=1)

labelled_df_clean = filter_job_ads(labelled_df)

labelled_df_clean["sentence"] = labelled_df_clean["sent"].apply(lambda x: x.text)

negative_sentences = get_negative_example_sentences(labelled_df_clean)

negative_df = pd.DataFrame(list(negative_sentences))
negative_df["label"] = 0
negative_df.columns = ["span", "label"]

positive_df = labelled_df_clean[["span"]]
positive_df["label"] = 1

training_ml_df = pd.concat([positive_df, negative_df])

# Splitting the dataset into training, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(
        training_ml_df.drop(["label"], axis=1),
        training_ml_df["label"],
        test_size=0.4,
        random_state=SEED,
    )
X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=SEED
    )

In [ ]:
input_df = pd.concat([X_train, y_train], axis=1)
input_df = input_df[input_df['label']==1]

In [ ]:
len(input_df)

In [ ]:
input_sentences = input_df['span'].tolist()

In [ ]:
lookup = pd.read_csv(PROJECT_DIR / "inputs/keyword_lookup - v3.csv")
lookup.head(20)

In [ ]:
targets = lookup['target_phrase'].tolist()

In [ ]:
target_embeddings = model.encode(targets, show_progress_bar=True)

In [ ]:
lookup['embeddings'] = target_embeddings.tolist()

In [ ]:
lookup.head()

In [ ]:
def get_n_most_similar_phrases(input_sentence, 
                                 lookup,
                                 model,
                                 n: int = 3):
    most_similar_phrases = {}
    
    input_embedding = model.encode(input_sentence)
    
    similarities = [cosine_similarity([input_embedding], [embed])[0][0] for embed in lookup['embeddings'].apply(pd.Series).values]
    
    top_indices = np.argsort(similarities)[::-1][:n]
    
    similar_phrases = lookup.iloc[top_indices]
    similar_phrases['similarity'] = [similarities[i] for i in top_indices]
    
    most_similar_phrases[input_sentence] = similar_phrases[['dimension', 'subcategory', 'target_phrase', 'similarity']]
        
    return most_similar_phrases

In [ ]:
# for _, row in X_train.iterrows():
#     output = get_n_most_similar_phrases(row['span'], lookup, model, 1)
#     output['span'] = row['span']
    

In [ ]:
for sentence in input_sentences[0:20]:
    print(f"Sentence: {sentence}")
    print(get_n_most_similar_phrases(sentence, lookup, model, 3))